In [12]:
import yfinance as yf
import pandas as pd 
from pandas.tseries.offsets import MonthEnd

In [13]:
# URL for the regularly updated Wikipedia page containing the current S&P 500 constituents
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies#S&P_500_component_stocks"
tickers = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})[0].Symbol

In [14]:
# URL for the Wikipedia page containing historical S&P 500 additions and removals
url_changes = "https://en.wikipedia.org/wiki/Historical_components_of_the_S&P_500"
changes = pd.read_html(url_changes, storage_options={"User-Agent": "Mozilla/5.0"})[0]

In [15]:
# Convert the Effective Date column to datetime format
changes[("Effective Date")] = pd.to_datetime(changes[("Effective Date", "Effective Date")])

In [16]:
# Remove changes that occurred before 2024
changes = changes[changes[("Effective Date", "Effective Date")] >= "2024-01-01"]

In [17]:
# Remove stocks that were added to the S&P 500 from 2024 onwards
tickers = tickers[~(tickers.isin(changes.Added.Ticker))]

In [18]:
# Get the tickers of stocks that were removed from the S&P 500 from 2024 onwards
tickers_rem = changes.Removed.Ticker

In [19]:
# Add stocks removed since 2024 to the current list of S&P 500 tickers
tickers = pd.concat([tickers, tickers_rem])

In [20]:
# Remove duplicate tickers and missing values
tickers.drop_duplicates(inplace=True)
tickers.dropna(inplace=True)

In [ ]:
#Variable to set the start date of the DataFrame
start = "2024-01-01"

In [ ]:
#empty lists to store price and symbols
prices, symbols = [], []

#Stores price data in prices list
for symbol in tickers:
    df = yf.download(symbol, start=start)["Close"]
    if not df.empty:
        prices.append(df)
        symbols.append(symbol)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%********

In [ ]:
#Stores Prices in a DataFrame
all_prices = pd.concat(prices, axis=1)

In [ ]:
all_prices

Ticker,MMM,AOS,ABT,ABBV,ACN,ADBE,AMD,AES,AFL,A,...,BBWI,AAL,ETSY,BIO,RHI,ILMN,XRAY,VFC,WHR,ZION
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-02,86.408249,77.233475,103.769409,145.533875,329.828705,580.070007,138.580002,16.883278,78.544304,136.120483,...,41.190311,13.44,81.080002,327.989990,75.034508,133.861862,33.107227,17.657448,106.673775,40.303596
2024-01-03,84.672218,74.644485,103.457703,146.116684,321.272095,571.789978,135.320007,16.525360,78.459419,128.674332,...,40.122574,12.95,79.129997,318.859985,73.143608,126.789886,32.354576,16.578121,102.307869,38.349594
2024-01-04,84.970741,75.279877,104.836876,147.027298,320.482941,567.049988,136.009995,16.472980,77.620041,128.517395,...,39.984516,13.09,76.290001,316.480011,72.411659,127.850197,33.153687,16.038454,102.559593,38.933975
2024-01-05,85.300636,74.957451,104.666855,147.646484,320.036133,564.599976,138.580002,16.411875,78.016144,128.085724,...,41.153492,13.60,75.639999,316.380005,72.237389,126.926071,32.874924,16.000586,104.981224,40.221424
2024-01-08,85.512749,75.175560,106.178276,146.999954,323.582397,580.549988,146.179993,16.647575,78.157616,130.852280,...,42.018715,14.58,77.760002,321.980011,72.542366,130.019455,33.358109,16.407701,105.840530,40.668835
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-08-17,180.210007,61.400002,110.370003,250.330002,169.979996,254.039993,506.000000,14.760000,120.373764,148.449997,...,18.618452,14.43,78.410004,355.230011,42.470001,193.139999,11.190000,14.480000,39.590000,71.580002
2026-08-18,180.940002,60.860001,112.680000,258.920013,172.929993,263.140015,484.390015,14.740000,121.120003,148.440002,...,19.835928,14.05,79.959999,352.690002,43.759998,188.289993,11.110000,14.310000,39.270000,70.930000
2026-08-19,180.660004,63.799999,114.430000,265.970001,183.169998,272.470001,466.420013,14.700000,116.510002,155.410004,...,19.469694,13.86,80.459999,364.260010,43.540001,205.000000,11.030000,14.370000,40.090000,68.290001


In [ ]:
#Calculates monthly returns via grouping daily returns into respective month-ends 
all_mtl_rtn = all_prices.pct_change(fill_method=None).resample("ME").agg(lambda x: (x + 1).prod() - 1)

In [43]:
all_mtl_rtn

Ticker,MMM,AOS,ABT,ABBV,ACN,ADBE,AMD,AES,AFL,A,...,BBWI,AAL,ETSY,BIO,RHI,ILMN,XRAY,VFC,WHR,ZION
Date,,,,,,,,,,,,,,,,,,,,,
2024-01-31,-0.142273,-0.043279,0.035017,0.038485,0.052718,0.065009,0.210059,-0.128656,0.012728,-0.062342,...,-0.046704,0.058780,-0.179082,-0.021647,-0.076298,0.039241,-0.024698,-0.117426,-0.108869,-0.050747
2024-02-29,-0.007418,0.068161,0.048519,0.070864,0.029955,-0.093075,0.148130,-0.088729,-0.036616,0.055803,...,0.075897,0.101897,0.077073,0.015550,0.017554,-0.022236,-0.059568,-0.007290,-0.003242,-0.049039
2024-03-31,0.151433,0.079131,-0.041976,0.034365,-0.075164,-0.099379,-0.062536,0.179606,0.063413,0.059333,...,0.094530,-0.021046,-0.041428,0.061342,-0.013931,-0.017950,0.020631,-0.055919,0.113977,0.100685
2024-04-30,0.088205,-0.070425,-0.063034,-0.098523,-0.128487,-0.082778,-0.122500,0.008091,-0.025739,-0.056679,...,-0.091963,-0.119870,-0.000728,-0.220083,-0.127901,-0.103918,-0.095812,-0.187744,-0.207055,-0.060369
2024-05-31,0.044819,0.009657,-0.035670,-0.008608,-0.061879,-0.039042,0.053795,0.206145,0.080518,-0.048380,...,0.143549,-0.148779,-0.075724,0.063429,-0.063653,-0.152540,-0.066644,0.065811,-0.000875,0.068935
2024-06-30,0.020471,-0.022238,0.016831,0.063756,0.074817,0.249078,-0.028101,-0.186197,-0.006231,-0.005981,...,-0.244914,-0.014783,-0.070742,-0.047933,-0.003892,0.028986,-0.105013,0.023348,0.098570,0.004168
2024-07-31,0.248165,0.043772,0.024945,0.090382,0.094460,-0.007002,-0.109303,0.012521,0.067966,0.092821,...,-0.058899,-0.060900,0.104442,0.238915,0.003282,0.174555,0.089522,0.256296,-0.002250,0.191376
2024-08-31,0.061671,-0.015522,0.069190,0.059303,0.034269,0.041258,0.028239,-0.027644,0.162603,0.010750,...,-0.158046,-0.001880,-0.154283,-0.003074,-0.015277,0.071778,-0.068165,0.073703,0.000612,-0.032085
2024-09-30,0.014923,0.072981,0.006533,0.005960,0.033718,-0.098588,0.104470,0.171045,0.013048,0.038903,...,0.037711,0.058380,0.007987,-0.008123,0.075634,-0.007534,0.076485,0.101189,0.066906,-0.047216


In [44]:
all_mtl_rtn_12 = all_mtl_rtn.rolling(12).agg(lambda x: (x + 1).prod() - 1)

In [47]:
all_mtl_rtn_12.dropna(inplace=True)
all_mtl_rtn_12

Ticker,MMM,AOS,ABT,ABBV,ACN,ADBE,AMD,AES,AFL,A,...,BBWI,AAL,ETSY,BIO,RHI,ILMN,XRAY,VFC,WHR,ZION
Date,,,,,,,,,,,,,,,,,,,,,
2024-12-31,0.452211,-0.148924,0.050225,0.152521,0.030339,-0.233403,-0.128374,-0.307174,0.268851,-0.025066,...,-0.115194,2.968751e-01,-0.347681,0.001585,-0.156625,-0.001732,-0.453423,0.175816,-0.003278,0.274060
2025-01-31,0.996193,-0.117836,0.153664,0.159361,0.075550,-0.291900,-0.308546,-0.309627,0.300627,0.172724,...,-0.099616,1.890373e-01,-0.175030,0.124622,-0.160436,-0.045824,-0.416551,0.612243,0.025967,0.431484
2025-02-28,1.059806,-0.184192,0.186973,0.230585,-0.054610,-0.217249,-0.481328,-0.201771,0.384038,-0.062263,...,-0.189821,-8.482142e-02,-0.285953,-0.186326,-0.240095,-0.347603,-0.480379,0.559045,0.015040,0.416905
2025-03-31,0.693649,-0.256750,0.190861,0.192489,-0.084720,-0.239933,-0.430772,-0.274849,0.321954,-0.190491,...,-0.380538,-3.127036e-01,-0.313446,-0.295805,-0.288569,-0.406048,-0.535660,0.032920,-0.193190,0.187721
2025-04-30,0.472118,-0.165626,0.258574,0.243160,0.011829,-0.189810,-0.385339,-0.420829,0.326249,-0.208985,...,-0.313526,-2.635085e-01,-0.366827,-0.095162,-0.337515,-0.351704,-0.522205,-0.026583,-0.138869,0.140060
2025-05-31,0.512089,-0.216838,0.333377,0.196173,0.142278,-0.066710,-0.336549,-0.506815,0.175826,-0.135397,...,-0.446723,-7.826100e-03,-0.127934,-0.208917,-0.259203,-0.189272,-0.411486,-0.042100,-0.098806,0.133054
2025-06-30,0.520610,-0.183333,0.335161,0.121520,0.002619,-0.303596,-0.125208,-0.368148,0.205139,-0.082866,...,-0.213405,-9.708708e-03,-0.149542,-0.116400,-0.333293,-0.085936,-0.339948,-0.111149,0.065415,0.237469
2025-07-31,0.194109,-0.151218,0.214038,0.056443,-0.177254,-0.351600,0.220307,-0.219952,0.063173,-0.181802,...,-0.192074,7.988716e-02,-0.105465,-0.284933,-0.402493,-0.162235,-0.454077,-0.294289,-0.125699,0.072285
2025-08-31,0.177723,-0.131746,0.193692,0.110105,-0.225750,-0.379015,0.094709,-0.162777,-0.011130,-0.114003,...,-0.025506,2.589454e-01,-0.037756,-0.116922,-0.376792,-0.239269,-0.414551,-0.151496,-0.010425,0.208629
